In [21]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/processed/urbanev_features.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

columns_to_exclude = [ 'timestamp', 'target_occ', 'target_vol', 'target_occ_ratio']

df['zone_id'] = df['zone_id'].astype('category')

print(f"Total dataset shape: {df.shape}")

Total dataset shape: (1148126, 34)


In [23]:
# Define the split boundary (Start of the 6th month)
split_date = '2023-02-01'

# Chronological split
train_df = df[df['timestamp'] < split_date].copy()
test_df = df[df['timestamp'] >= split_date].copy()

# Construct X and y
X_train = train_df.drop(columns=columns_to_exclude)
y_train = train_df['target_occ']

X_test = test_df.drop(columns=columns_to_exclude)
y_test = test_df['target_occ']

print(f"Training Set (First 5 Months): X={X_train.shape}, y={y_train.shape}")
print(f"Testing Set (Last 1 Month): X={X_test.shape}, y={y_test.shape}")
print(X_train.columns.tolist())

Training Set (First 5 Months): X=(963601, 30), y=(963601,)
Testing Set (Last 1 Month): X=(184525, 30), y=(184525,)
['zone_id', 'charge_count', 'occupancy', 'occ_ratio', 'volume', 'duration', 's_price', 'e_price', 'hour', 'day_of_week', 'is_weekend', 'occ_lag_1', 'occ_ratio_lag_1', 'vol_lag_1', 'duration_lag_1', 'occ_lag_2', 'occ_ratio_lag_2', 'vol_lag_2', 'duration_lag_2', 'occ_lag_24', 'occ_ratio_lag_24', 'vol_lag_24', 'duration_lag_24', 'occ_lag_168', 'occ_ratio_lag_168', 'vol_lag_168', 'duration_lag_168', 'occ_rolling_3h', 'occ_rolling_6h', 'price_spread']


In [24]:
# Dictionary to store results for the final table
results_dict = {}

def evaluate_and_store(model_name, model, X_train, y_train, X_test, y_test, fit_time):
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    train_mae = mean_absolute_error(y_train, y_pred_train)
    train_r2 = r2_score(y_train, y_pred_train)
    
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_r2 = r2_score(y_test, y_pred_test)
    
    # Store results
    results_dict[model_name] = {
        'Train RMSE': round(train_rmse, 4),
        'Train MAE': round(train_mae, 4),
        'Train R²': round(train_r2, 4),
        'Test RMSE': round(test_rmse, 4),
        'Test MAE': round(test_mae, 4),
        'Test R²': round(test_r2, 4),
        'Training Time (s)': round(fit_time, 2)
    }
    print(f"--- {model_name} Evaluated ---")

In [25]:

baseline_pred = X_test["occupancy"]

print(
    "Baseline MAE:",
    mean_absolute_error(y_test, baseline_pred)
)

print(
    "Baseline R2:",
    r2_score(y_test, baseline_pred)
)

Baseline MAE: 1.2986695569706002
Baseline R2: 0.9758798025335753


In [ ]:
baseline24 = X_test["occ_lag_24"]

print(mean_absolute_error(y_test, baseline24))
print(r2_score(y_test, baseline24))

3.198764395068419
0.9164708184295792


In [27]:
# ----------------------------------------
# 1. LightGBM
# ----------------------------------------
print("Training LightGBM...")
start_time = time.time()
lgb_model = lgb.LGBMRegressor(random_state=42, n_estimators=1000, n_jobs=-1, learning_rate=0.05)

# Fit with early stopping
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)
lgb_time = time.time() - start_time
evaluate_and_store('LightGBM', lgb_model, X_train, y_train, X_test, y_test, lgb_time)

Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6557
[LightGBM] [Info] Number of data points in the train set: 963601, number of used features: 30
[LightGBM] [Info] Start training from score 17.926186
--- LightGBM Evaluated ---


In [28]:
# ----------------------------------------
# 2. XGBoost
# ----------------------------------------
print("Training XGBoost...")
start_time = time.time()
# enable_categorical=True and tree_method='hist' are required for XGBoost to handle the categorical zone_id
xgb_model = xgb.XGBRegressor(
    random_state=42, 
    n_estimators=1000, 
    enable_categorical=True, 
    early_stopping_rounds=50,
    tree_method='hist',
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
xgb_time = time.time() - start_time
evaluate_and_store('XGBoost', xgb_model, X_train, y_train, X_test, y_test, xgb_time)

Training XGBoost...
--- XGBoost Evaluated ---


In [29]:
# ----------------------------------------
# 3. CatBoost
# ----------------------------------------
print("Training CatBoost...")
start_time = time.time()
cat_model = CatBoostRegressor(
    random_state=42, 
    iterations=1000, 
    cat_features=['zone_id'], 
    thread_count=-1
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=False
)
cat_time = time.time() - start_time
evaluate_and_store('CatBoost', cat_model, X_train, y_train, X_test, y_test, cat_time)

Training CatBoost...
--- CatBoost Evaluated ---


In [30]:
# Create and display the comparison table
results_df = pd.DataFrame.from_dict(results_dict, orient='index')

# Sort by Test R² (highest is best) to immediately see the winner
results_df = results_df.sort_values(by='Test R²', ascending=False)

display(results_df)

,Train RMSE,Train MAE,Train R²,Test RMSE,Test MAE,Test R²,Training Time (s)
CatBoost,3.0050,1.2817,0.9797,2.0327,1.0397,0.9904,107.08
LightGBM,2.8951,1.2444,0.9811,2.2055,1.0843,0.9887,12.91
XGBoost,3.0285,1.3278,0.9794,2.4013,1.1883,0.9866,3.83
